# 1. install packages

In [1]:
%%capture
!pip install open-clip-torch timm transformers ftfy regex -q

# 2. Import library

In [2]:
import torch
import numpy as np
from PIL import Image
from pathlib import Path
import json
import time
from tqdm.auto import tqdm
import gc
import open_clip
import shutil
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
import os

# 3. Embedder

In [3]:
class Embedder:
    """
    CLIP Embedder được đơn giản hóa.
    Chỉ có nhiệm vụ duy nhất là encode một batch ảnh được đưa cho.
    Không còn logic padding.
    """
    
    def __init__(self, device, model_name, pretrained, tokenizer_model, use_multi_gpu=False):
        self.device = device
        self.model_name = model_name
        self.pretrained = pretrained
        self.tokenizer_model = tokenizer_model
        self.use_multi_gpu = use_multi_gpu
        self._load_model()
        self._setup_multi_gpu()
        
    def _load_model(self):
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(
            self.model_name, pretrained=self.pretrained, device=self.device)
        self.model.eval()
    
    def _setup_multi_gpu(self):
        if self.use_multi_gpu and torch.cuda.device_count() > 1:
            self.model = torch.nn.DataParallel(self.model)
            # self.model = self.model.to(self.device)
            print(f"Multi-GPU enabled: {torch.cuda.device_count()} GPUs")
    
    def _get_actual_model(self):
        return self.model.module if hasattr(self.model, 'module') else self.model
    
    def encode_images_batch(self, images_batch):
        """
        Encode một batch ảnh PIL. Trả về mảng numpy của các embedding.
        """
        if not images_batch:
            return None
        
        try:
            image_tensors = torch.stack([self.preprocess(img) for img in images_batch]).to(self.device)
            
            with torch.cuda.amp.autocast(enabled=self.device.type == 'cuda'):
                with torch.no_grad():
                    batch_embeddings = self._get_actual_model().encode_image(image_tensors)
                    batch_embeddings_norm = torch.nn.functional.normalize(batch_embeddings, p=2, dim=-1)
            
            results = batch_embeddings_norm.cpu().numpy()
            
            del image_tensors, batch_embeddings, batch_embeddings_norm
            if self.device.type == 'cuda':
                torch.cuda.empty_cache()
            
            return results
            
        except Exception as e:
            print(f"Batch encoding failed: {e}")
            if self.device.type == 'cuda':
                torch.cuda.empty_cache()
            return None

    def cleanup(self):
        gc.collect()
        if self.device.type == 'cuda':
            torch.cuda.empty_cache()

# 4. Keyframe Extractor

In [ ]:
from collections import defaultdict
from pathlib import Path
import os
import shutil
from tqdm import tqdm
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image

class KeyframeExtractor:
    """
    Main Video Processor với logic tổng hợp batch thông minh.
    Hàm trích xuất keyframe đã được tích hợp thành một phương thức nội bộ.
    """
    
    def __init__(self, miniembedder, output_dir, batch_size, similarity_threshold, keyframe_progress=None):
        self.miniembedder = miniembedder
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True, parents=True)
        self.batch_size = batch_size
        self.similarity_threshold = similarity_threshold
        self.keyframe_progress = keyframe_progress
        
        print(f"📁 Output directory: {self.output_dir}")
        print(f"📦 Batch size: {self.batch_size}")
        if self.keyframe_progress:
            print(f"Progress control: Start={self.keyframe_progress.get('start', 'N/A')}, End={self.keyframe_progress.get('end', 'N/A')}")
        else:
            print("No progress control. Will process all videos.")

    def _extract_keyframes(self, embeddings):
        """
        [PHƯƠNG THỨC NỘI BỘ]
        Trích xuất keyframe bằng cosine similarity.
        Sử dụng self.similarity_threshold đã được thiết lập.
        """
        if embeddings is None or len(embeddings) <= 1:
            return [0] if embeddings is not None and len(embeddings) > 0 else []
        
        keyframes = [0]
        last_keyframe_embedding = embeddings[0]
        
        for i in range(1, len(embeddings)):
            similarity = cosine_similarity([last_keyframe_embedding], [embeddings[i]])[0][0]
            if similarity < self.similarity_threshold:
                keyframes.append(i)
                last_keyframe_embedding = embeddings[i]
                
        return keyframes

    def save_keyframe_images(self, video_name, keyframe_paths):
        """Sao chép các file ảnh keyframe đã xác định vào thư mục output."""
        video_dir = self.output_dir / video_name
        video_dir.mkdir(exist_ok=True, parents=True)
        
        for source_path in keyframe_paths:
            try:
                shutil.copy(source_path, video_dir / Path(source_path).name)
            except Exception as e:
                print(f"   ⚠️ Could not copy file {Path(source_path).name}: {e}")
        
        print(f"   💾 Saved {len(keyframe_paths)} keyframe images to {video_name}/")

    def process_dataset(self, dataset_path, embedding_file="all_embeddings.pkl"):
        """
        Thực hiện workflow xử lý dataset:
        1. Giai đoạn 1: Quét, tổng hợp batch và embedding.
        2. Giai đoạn 2: Trích xuất keyframe và lưu kết quả.
        """
        all_video_folders = sorted([
            os.path.join(dataset_path, folder_name) for folder_name in os.listdir(dataset_path)
        ])
        
        start_index = self.keyframe_progress.get('start', 0)
        end_index = self.keyframe_progress.get('end', len(all_video_folders)) 
        videos_to_process = all_video_folders[start_index:end_index]
        
        if not videos_to_process:
            print("No videos to process in the specified range.")
            return

        print(f"Processing {len(videos_to_process)} videos from index {start_index} to {end_index}.")
        
        # --- GIAI ĐOẠN 1: TỔNG HỢP BATCH VÀ EMBEDDING ---
        print("\n--- STAGE 1: Aggregating batches and embedding ---")
        
        image_buffer = []
        metadata_buffer = []
        all_results = defaultdict(list) 

        for video_folder in tqdm(videos_to_process, desc="Aggregating Videos"):
            video_name = os.path.basename(video_folder)
            frame_paths = sorted(
                [
                    os.path.join(video_folder, file)
                    for file in os.listdir(video_folder)
                    if file.endswith('.jpg')
                ],
                key=lambda x: int(Path(x).stem)
            )
            # frame_paths = sorted(video_folder.glob("*.jpg"), key=lambda x: int(x.stem))
            
            for path in frame_paths:
                image_buffer.append(Image.open(path).convert("RGB"))
                metadata_buffer.append({'video_name': video_name, 'path': path})

                if len(image_buffer) >= self.batch_size:
                    batch_embeddings = self.miniembedder.encode_images_batch(image_buffer)
                    if batch_embeddings is not None:
                        for i in range(self.batch_size):
                            meta = metadata_buffer[i]
                            all_results[meta['video_name']].append({'embedding': batch_embeddings[i], 'path': meta['path']})
                    image_buffer = image_buffer[self.batch_size:]
                    metadata_buffer = metadata_buffer[self.batch_size:]

        if image_buffer:
            print(f"Flushing final batch of {len(image_buffer)} images...")
            batch_embeddings = self.miniembedder.encode_images_batch(image_buffer)
            if batch_embeddings is not None:
                for i in range(len(image_buffer)):
                    meta = metadata_buffer[i]
                    all_results[meta['video_name']].append({'embedding': batch_embeddings[i], 'path': meta['path']})

        print("✅ Stage 1 Completed: All frames have been embedded.")
        self.miniembedder.cleanup()
        
        # Save embeddings to pickle file
        import pickle
        embeddings_file = self.output_dir / embedding_file
        with open(embeddings_file, 'wb') as f:
            pickle.dump(dict(all_results), f)
        print(f"💾 Saved all embeddings to {embeddings_file}")
        
        # --- GIAI ĐOẠN 2: TRÍCH XUẤT KEYFRAME VÀ LƯU ---
        print("\n--- STAGE 2: Extracting keyframes and saving ---")
        
        for video_name, results in tqdm(all_results.items(), desc="Extracting Keyframes"):
            results.sort(key=lambda x: int(Path(x['path']).stem))
            
            embeddings = np.array([res['embedding'] for res in results])
            paths = [res['path'] for res in results]
            
            # Gọi phương thức nội bộ
            keyframe_indices = self._extract_keyframes(embeddings)
            keyframe_paths = [paths[i] for i in keyframe_indices]
            
            self.save_keyframe_images(video_name, keyframe_paths)
            
        print("✅ Stage 2 Completed: All keyframes have been saved.")

# 5. Check GPU

In [6]:
# Check GPU availability
print("🔍 Checking GPU availability...")

try:
    if torch.cuda.is_available():
        device = torch.device("cuda")
        gpu_count = torch.cuda.device_count()
        
        print(f"🔢 GPU Count: {gpu_count}")
        
        # Multi-GPU strategy
        if gpu_count > 1:
            print(f"Multi-GPU detected! {gpu_count} GPUs available")
            use_multi_gpu = True
        else:
            print(f"Single GPU: {torch.cuda.get_device_name(0)}")
            use_multi_gpu = False
        
        # Clear GPU cache
        torch.cuda.empty_cache()
        
    else:
        device = torch.device("cpu")
        gpu_count = 0
        use_multi_gpu = False
        print("GPU not available!")        
except Exception as e:
    print(f"GPU setup failed: {e}")
    device = torch.device("cpu")
    gpu_count = 0
    use_multi_gpu = False

print(f"Using device: {device}")
print("GPU setup completed!")

🔍 Checking GPU availability...
🔢 GPU Count: 1
Single GPU: Tesla P100-PCIE-16GB
Using device: cuda
GPU setup completed!


# 6. Configs and run

In [7]:
DATASET_PATH = "/kaggle/input/lucier-f1"
OUTPUT_PATH = "/kaggle/working/lucifer-kf1" 

# Processing Parameters
SIMILARITY_THRESHOLD = 0.95
MINI_EMBEDDER_MODEL_NAME = "ViT-L-14"
MINI_EMBEDDER_PRETRAINED = "commonpool_xl_clip_s13b_b90k"
MINI_EMBEDDER_TOKENIZER_MODEL = "ViT-L-14"
MINI_EMBEDDER_BATCH_SIZE = 512

try:
    miniembedder = Embedder(
        device=device, 
        model_name=MINI_EMBEDDER_MODEL_NAME,
        pretrained=MINI_EMBEDDER_PRETRAINED,
        tokenizer_model=MINI_EMBEDDER_TOKENIZER_MODEL,
        use_multi_gpu=use_multi_gpu
    )
    print(f"✅ Mini embedder ({MINI_EMBEDDER_MODEL_NAME}) initialized successfully")
    
except Exception as e:
    print(f"Failed to initialize embedder: {e}")
    raise e

# --- TÙY CHỌN 1: Xử lý một phần của dataset ---
# keyframe_progress_config = {
#     'start': 0,
#     'end': 4 
# }

# --- TÙY CHỌN 2: Xử lý toàn bộ dataset ---
# Đặt config là None để xử lý tất cả.
keyframe_progress_config = {}


processor = KeyframeExtractor(
    miniembedder=miniembedder,
    output_dir=OUTPUT_PATH,
    batch_size=MINI_EMBEDDER_BATCH_SIZE,
    similarity_threshold=SIMILARITY_THRESHOLD,
    keyframe_progress=keyframe_progress_config
)

processor.process_dataset(DATASET_PATH, "lucifer_kf1.pkl")

open_clip_pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

✅ Mini embedder (ViT-L-14) initialized successfully
📁 Output directory: /kaggle/working/lucifer-kf5
📦 Batch size: 512
Progress control: Start=0, End=4
Processing 4 videos from index 0 to 4.

--- STAGE 1: Aggregating batches and embedding ---


Aggregating Videos:   0%|          | 0/4 [00:00<?, ?it/s]

/tmp/ipykernel_36/640857946.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.device.type == 'cuda'):


Flushing final batch of 300 images...
✅ Stage 1 Completed: All frames have been embedded.

--- STAGE 2: Extracting keyframes and saving ---


Extracting Keyframes:   0%|          | 0/4 [00:00<?, ?it/s]

   💾 Saved 269 keyframe images to L25_V001/
   💾 Saved 259 keyframe images to L25_V002/
   💾 Saved 247 keyframe images to L25_V003/
   💾 Saved 264 keyframe images to L25_V004/
✅ Stage 2 Completed: All keyframes have been saved.
